# HippocampalPlaceCellExample

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `decoding_2d`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/HippocampalPlaceCellExample.md`


Notebook source link: [HippocampalPlaceCellExample.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/HippocampalPlaceCellExample.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "HippocampalPlaceCellExample"
FAMILY = "decoding_2d"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"HippocampalPlaceCellExample: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"HippocampalPlaceCellExample: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"HippocampalPlaceCellExample: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"HippocampalPlaceCellExample: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# HippocampalPlaceCellExample: MATLAB-gold parity workflow.
from pathlib import Path
from scipy.io import loadmat
from nstat.compat.matlab import DecodingAlgorithms


def resolve_repo_root() -> Path:
    candidates = [Path.cwd().resolve()]
    candidates.append(candidates[0].parent)
    candidates.append(candidates[1].parent)
    for root in candidates:
        if (root / "tests" / "parity" / "fixtures" / "matlab_gold").exists():
            return root
    return candidates[0]


repo_root = resolve_repo_root()
fixture_path = repo_root / "tests" / "parity" / "fixtures" / "matlab_gold" / "HippocampalPlaceCellExample_gold.mat"
m = loadmat(fixture_path)

spike_counts = np.asarray(m["spike_counts_pc"], dtype=float)
tuning_curves = np.asarray(m["tuning_curves"], dtype=float)
expected_weighted = np.asarray(m["expected_decoded_weighted"], dtype=float).reshape(-1)

decoded_weighted = DecodingAlgorithms.decodeWeightedCenter(spike_counts, tuning_curves)
abs_err = np.abs(decoded_weighted - expected_weighted)
mae = float(np.mean(abs_err))
max_err = float(np.max(abs_err))

n_time = decoded_weighted.size
n_states = tuning_curves.shape[1]
time = np.arange(n_time, dtype=float)
x_true = expected_weighted / max(float(n_states - 1), 1.0)
y_true = 0.5 + 0.35 * np.sin(2.0 * np.pi * np.arange(n_time) / max(float(n_time), 1.0))
x_decoded = decoded_weighted / max(float(n_states - 1), 1.0)
y_decoded = 0.5 + 0.35 * np.sin(2.0 * np.pi * np.arange(n_time) / max(float(n_time), 1.0))

example_cell = 24
rep = np.clip(spike_counts[example_cell].astype(int), 0, 4)
spike_x = np.repeat(x_true, rep)
spike_y = np.repeat(y_true, rep)

fig1, ax = plt.subplots(1, 1, figsize=(7.4, 4.8))
ax.plot(x_true, y_true, "b", linewidth=1.0, label="animal path")
if spike_x.size:
    ax.plot(spike_x, spike_y, "r.", markersize=3, label="spike positions")
ax.set_title("Example data: trajectory and spike locations")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal", adjustable="box")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

fig2, axes = plt.subplots(3, 4, figsize=(10.8, 7.2))
for i, ax in enumerate(axes.ravel(), start=0):
    if i >= tuning_curves.shape[0]:
        ax.axis("off")
        continue
    field = tuning_curves[i].reshape(5, 8)
    ax.imshow(field, origin="lower", cmap="jet", aspect="auto")
    ax.set_title(f"Cell {i+1}", fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])
fig2.suptitle("Place fields (MATLAB-gold tuning curves)", y=0.99, fontsize=11)
plt.tight_layout()
plt.show()

fig3, axes = plt.subplots(2, 1, figsize=(9.6, 6.4), sharex=True)
axes[0].plot(time, expected_weighted, "k", linewidth=1.1, label="MATLAB weighted")
axes[0].plot(time, decoded_weighted, "g--", linewidth=0.9, label="Python weighted")
axes[0].set_title("Weighted-center decoding")
axes[0].set_ylabel("state index")
axes[0].legend(loc="upper right")

axes[1].plot(time, abs_err, "m", linewidth=1.0)
axes[1].set_title("Absolute decode error")
axes[1].set_xlabel("time bin")
axes[1].set_ylabel("|error|")
plt.tight_layout()
plt.show()

assert decoded_weighted.shape == expected_weighted.shape
assert mae < 1e-10
assert max_err < 1e-10
assert spike_x.size > 0

CHECKPOINT_METRICS = {
    "weighted_mae": float(mae),
    "weighted_max_err": float(max_err),
    "spike_points": float(spike_x.size),
}
CHECKPOINT_LIMITS = {
    "weighted_mae": (0.0, 1e-10),
    "weighted_max_err": (0.0, 1e-10),
    "spike_points": (1.0, 50000.0),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
